# Patrón Creacional: Factory Method

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Factory Method** define una **interfaz para crear objetos**, pero deja que
las **subclases decidan** qué clase concreta instanciar. El método de fábrica es
**sobrescribible**, no un simple `if/elif`.

### ¿Qué problema resuelve en la banca?
Un banco ofrece varios **tipos de cuenta** (ahorros, corriente, nómina) y cada una
calcula distinto sus intereses y comisiones. Necesitamos **crear la cuenta adecuada**
sin llenar el código de condicionales que haya que tocar cada vez que aparece un nuevo
producto financiero.

## Código *sin patrón* (el problema es evidente)
Una función con `if/elif` que crea el objeto según un string. Cada nuevo producto obliga
a **modificar** esta función.

In [1]:
class CuentaAhorros:
    def resumen(self):
        return "Cuenta de ahorros: interes 2%, sin cuota de manejo"

class CuentaCorriente:
    def resumen(self):
        return "Cuenta corriente: permite sobregiro, cuota de manejo mensual"


def crear_cuenta_sin_patron(tipo):
    if tipo == "ahorros":
        return CuentaAhorros()
    elif tipo == "corriente":
        return CuentaCorriente()
    else:
        raise ValueError("Tipo de cuenta no soportado")


for t in ["ahorros", "corriente"]:
    print(crear_cuenta_sin_patron(t).resumen())
print(">> Problema: para agregar 'nomina' hay que MODIFICAR la funcion (viola Open/Closed).")

Cuenta de ahorros: interes 2%, sin cuota de manejo
Cuenta corriente: permite sobregiro, cuota de manejo mensual
>> Problema: para agregar 'nomina' hay que MODIFICAR la funcion (viola Open/Closed).


### Análisis del problema
- La función concentra el conocimiento de **todos** los tipos de cuenta.
- Cada nuevo producto (nómina, AFC, CDT...) obliga a **editar** el `if/elif`: viola el
  principio **Open/Closed**.
- No hay un punto de extensión: no puedo agregar un producto sin tocar código existente.

## Código *con patrón* (problema resuelto)
Definimos un **creador abstracto** con el *factory method* `crear_cuenta()`. Cada
subclase (sucursal/creador concreto) decide qué **producto concreto** instanciar.
Agregar un producto = agregar una subclase, **sin tocar** las existentes.

In [2]:
from abc import ABC, abstractmethod


# --- Productos ---
class Cuenta(ABC):
    @abstractmethod
    def resumen(self) -> str: ...


class CuentaAhorros(Cuenta):
    def resumen(self) -> str:
        return "Cuenta de ahorros: interes 2%, sin cuota de manejo"


class CuentaCorriente(Cuenta):
    def resumen(self) -> str:
        return "Cuenta corriente: permite sobregiro, cuota de manejo mensual"


class CuentaNomina(Cuenta):
    def resumen(self) -> str:
        return "Cuenta nomina: sin cuota, ligada al pago de salario"


# --- Creador abstracto con el Factory Method ---
class CreadorCuenta(ABC):
    @abstractmethod
    def crear_cuenta(self) -> Cuenta:
        """Factory Method: las subclases deciden el producto concreto."""

    def abrir_cuenta(self) -> str:
        # Logica comun que USA el producto sin conocer su clase concreta
        cuenta = self.crear_cuenta()
        return "Abriendo -> " + cuenta.resumen()


# --- Creadores concretos ---
class CreadorAhorros(CreadorCuenta):
    def crear_cuenta(self) -> Cuenta:
        return CuentaAhorros()


class CreadorCorriente(CreadorCuenta):
    def crear_cuenta(self) -> Cuenta:
        return CuentaCorriente()


class CreadorNomina(CreadorCuenta):
    def crear_cuenta(self) -> Cuenta:
        return CuentaNomina()


creadores = [CreadorAhorros(), CreadorCorriente(), CreadorNomina()]
for creador in creadores:
    print(creador.abrir_cuenta())
print(">> Solucion: 'nomina' se agrego con una subclase nueva, sin modificar las demas.")

Abriendo -> Cuenta de ahorros: interes 2%, sin cuota de manejo
Abriendo -> Cuenta corriente: permite sobregiro, cuota de manejo mensual
Abriendo -> Cuenta nomina: sin cuota, ligada al pago de salario
>> Solucion: 'nomina' se agrego con una subclase nueva, sin modificar las demas.


### Verificación
- Existe un **método de fábrica** (`crear_cuenta`) **sobrescrito por subclases**, no un
  `if/elif`.
- Hay **3 productos concretos** (`CuentaAhorros`, `CuentaCorriente`, `CuentaNomina`) y
  sus creadores.
- Añadir `CuentaNomina` no requirió modificar los creadores anteriores: **Open/Closed** cumplido.

## UML del patrón Factory Method
```plantuml
@startuml
abstract class CreadorCuenta {
    + crear_cuenta() : Cuenta
    + abrir_cuenta()
}
CreadorCuenta <|-- CreadorAhorros
CreadorCuenta <|-- CreadorCorriente
CreadorCuenta <|-- CreadorNomina

interface Cuenta {
    + resumen()
}
Cuenta <|.. CuentaAhorros
Cuenta <|.. CuentaCorriente
Cuenta <|.. CuentaNomina

CreadorAhorros --> CuentaAhorros : crea
CreadorCorriente --> CuentaCorriente : crea
CreadorNomina --> CuentaNomina : crea
@enduml
```

## ¿Por qué Factory Method y no otro patrón?
- El problema es **qué clase concreta instanciar** dejando un **punto de extensión** por
  herencia. Eso es exactamente Factory Method.
- No es Builder: aquí no ensamblamos un objeto con muchas partes, solo elegimos el tipo.
- No es Abstract Factory: no creamos **familias** de productos relacionados, solo un
  producto por creador. Si más adelante cada producto trajera además su propia tarjeta y
  su propio contrato, ahí sí escalaríamos a Abstract Factory.
- Frente al `if/elif`, Factory Method respeta **Open/Closed**: nuevos productos entran
  como subclases sin tocar el código existente.